# ingest circuit csv file
- read file using spark dataframe reader api
- add metadata columns
 > source file and
 > ingestion timestamp
- wrrite to bronze delta table 

In [0]:
%run ../config/config-env

In [0]:
%run ../config/helper-function

In [0]:
source_file = f'{landing_folder_path}/races.csv'
table_name = f'{catalog_name}.{bronze_schema}.races'

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

race_schema = StructType([
        StructField('season',   IntegerType()),
        StructField('round',    IntegerType()),
        StructField('url',      StringType()),
        StructField('racename', StringType()),
        StructField('date',     DateType()),
        StructField('circuitId', StringType())
])

In [0]:
races_df =(
    spark.read.format('csv')
    .option('header', True)
    .schema(race_schema)
    .load(source_file)
)

In [0]:
display(races_df)

### step 2 metadata

In [0]:

races_final_df = add_ingestion_metadata(races_df)
                  

In [0]:
display(races_final_df)

write files to delta format

In [0]:
(
    races_final_df.write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))